In [37]:
import nflreadpy as nfl
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
import joblib


In [14]:
pbp = nfl.load_pbp([2023,2024,2025])
pbp = pbp.to_pandas()

In [ ]:
df = pbp.copy()

df = df[df["play_type"].isin(["run", "pass"])]
features = [
    "down",
    "ydstogo",
    "yardline_100",
    "score_differential",
    "qtr",
    "shotgun",
    "quarter_seconds_remaining"
]

target = "play_type"


df = df[features + [target]].dropna()


X = df[features]
#X = pd.get_dummies(X, columns=["posteam"])
#X = pd.get_dummies(X, columns=["defteam"])

y = df[target]

y = y.map({
    "run": 0,
    "pass": 1
})




X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [51]:
print(df["play_type"].value_counts(normalize=True))

play_type
pass    0.57381
run     0.42619
Name: proportion, dtype: float64


In [35]:
pd.crosstab(
    df["shotgun"],
    df["play_type"],
    normalize="index"
)

play_type,pass,run
shotgun,,
0.0,0.292674,0.707326
1.0,0.696334,0.303666


In [52]:


model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    random_state=42
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

accuracy = accuracy_score(y_test, preds)


print("Accuracy:", accuracy)

train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)

print("Train: " + str(train_acc))


print(confusion_matrix(y_test, preds,normalize="true"))

Accuracy: 0.7100019069412662
Train: 0.8948773569164025
[[0.62789661 0.37210339]
 [0.22858333 0.77141667]]


In [44]:
joblib.dump(model, "model.pkl")

['model.pkl']

NameError: name 'y_pred' is not defined

In [53]:
from sklearn.metrics import classification_report
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.67      0.63      0.65      8976
           1       0.73      0.77      0.75     12000

    accuracy                           0.71     20976
   macro avg       0.70      0.70      0.70     20976
weighted avg       0.71      0.71      0.71     20976



In [54]:
importances = forrest.feature_importances_
feature_shapes = pd.Series(importances, index=X_train.columns).sort_values(ascending=False)
print(feature_shapes)

ValueError: Length of values (8) does not match length of index (11)